In [25]:
from __future__ import annotations

import torch

In [26]:
# Mean absolute error over all per-atom force components (x, y, z counted separately)
def force_mae(pred: torch.Tensor, target: torch.Tensor) -> float:
    return (pred.detach() - target.detach()).abs().mean().item()

In [27]:
# ECE: compares predicted uncertainty to observed error, binned
def expected_calibration_error(uncertainties: torch.Tensor, errors: torch.Tensor, n_bins: int = 10) -> float:
    uncertainties = uncertainties.detach()
    errors = errors.detach()
    bin_edges = torch.linspace(uncertainties.min(), uncertainties.max(), n_bins + 1)
    ece = 0.0
    n = uncertainties.shape[0]
    for i in range(n_bins):
        mask = (uncertainties >= bin_edges[i]) & (uncertainties < bin_edges[i + 1])
        if mask.sum() == 0:
            continue
        bin_uncertainty = uncertainties[mask].mean()
        bin_error = errors[mask].mean()
        ece += (mask.sum().item() / n) * abs(bin_uncertainty - bin_error).item()
    return ece

In [28]:
# Spearman rank correlation between predicted uncertainty and actual error
def uncertainty_error_correlation(uncertainties: torch.Tensor, errors: torch.Tensor) -> float:
    u_rank = uncertainties.argsort().argsort().float()
    e_rank = errors.argsort().argsort().float()
    u_centered = u_rank - u_rank.mean()
    e_centered = e_rank - e_rank.mean()
    denom = (u_centered.norm() * e_centered.norm())
    if denom == 0:
        return 0.0
    return ((u_centered * e_centered).sum() / denom).item()